In [6]:
# 第一部分：导入必要的库
# ============================================

import numpy as np

import mindspore as ms
from mindspore import nn, Tensor
from mindspore.common.initializer import HeNormal
import mindspore.dataset as ds

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')


In [7]:
# 第二部分：可调参数配置区（自编码器降维 + 线性SVM 经典对照组）
# ============================================

# 数据相关参数（与量子实验保持一致）
SAMPLE_RATIO = 1            # 数据采样比例
TEST_SIZE = 0.2             # 测试集比例
PCA_COMPONENTS = 8          # 降维目标维度（与量子比特数一致）

# 降维方法
DIM_REDUCTION_METHOD = 'autoencoder'

# 自编码器训练参数（与实验三基线一致）
AUTOENCODER_EPOCHS = 50         # 自编码器训练轮数
AUTOENCODER_LR = 0.001          # 自编码器学习率
AUTOENCODER_HIDDEN_DIMS = [256, 64]  # 编码器隐藏层维度（输入784 -> 256 -> 64 -> 8）

# SVM 参数
SVM_KERNEL = 'linear'       # 核函数
SVM_C = 1.0                 # 正则化参数（默认值，不调优）

# MindSpore 配置
MS_MODE = ms.PYNATIVE_MODE
MS_DEVICE = "CPU"

# 随机种子（与量子实验一致）
SEED = 42

# 初始化设置
# ============================================
ms.set_context(mode=MS_MODE, device_target=MS_DEVICE)
ms.set_seed(SEED)
np.random.seed(SEED)


[WARNING] ME(36516:44148,MainProcess):2026-10-07-01:11:56.450.000 [mindspore\context.py:1412] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


In [8]:
# 第三部分：数据加载、自编码器降维与预处理（与实验三基线一致）
# ============================================

import os
import gzip
import struct
import urllib.request

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

_MNIST_FILES = {
    'train_images': ('train-images-idx3-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz'),
    'train_labels': ('train-labels-idx1-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-labels-idx1-ubyte.gz'),
    'test_images':  ('t10k-images-idx3-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-images-idx3-ubyte.gz'),
    'test_labels':  ('t10k-labels-idx1-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz'),
}

def _ensure_mnist_files(data_dir='./MNIST'):
    """下载/校验 MNIST 原始 IDX 文件（有本地缓存则不重复下载）"""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, (fname, url) in _MNIST_FILES.items():
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            print(f"下载 {fname} ...")
            try:
                urllib.request.urlretrieve(url, path)
            except Exception as e:
                raise RuntimeError(
                    f"下载 {fname} 失败：{e}。请确认本机可联网，"
                    f"或把原始电脑上的 ./MNIST 目录（含 4 个 .gz 文件）拷贝到本机后重试。"
                )
        paths[key] = path
    return paths

def _read_idx_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols)

def _read_idx_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

def load_mnist_data():
    """加载MNIST数据集（不依赖 torchvision，直接解析官方 IDX 文件，数据与 torchvision 一致）"""
    print("正在加载MNIST数据集...")
    paths = _ensure_mnist_files('./MNIST')

    train_images = _read_idx_images(paths['train_images'])   # (60000, 28, 28)
    train_labels = _read_idx_labels(paths['train_labels'])   # (60000,)
    test_images  = _read_idx_images(paths['test_images'])    # (10000, 28, 28)
    test_labels  = _read_idx_labels(paths['test_labels'])    # (10000,)

    def flatten(imgs):
        return imgs.astype(np.float32).reshape(len(imgs), -1) / 255.0

    # 合并（顺序与 torchvision 一致：先 60000 训练 + 后 10000 测试）
    all_images = np.concatenate([flatten(train_images), flatten(test_images)])
    all_labels = np.concatenate([train_labels, test_labels]).astype(np.int64)

    sampled_indices = np.random.choice(
        len(all_images),
        int(len(all_images) * SAMPLE_RATIO),
        replace=False
    )
    sampled_images = all_images[sampled_indices]
    sampled_labels = all_labels[sampled_indices]

    print(f"总样本数: {len(all_images)}, 采样后: {len(sampled_images)}")
    print(f"类别分布: {np.unique(sampled_labels, return_counts=True)}")

    return sampled_images, sampled_labels

class Autoencoder(nn.Cell):
    """自编码器模型（与实验三基线一致：784 -> 256 -> 64 -> 8，HeNormal，解码器无 Sigmoid）"""
    def __init__(self, input_dim=784, hidden_dims=[256, 64], latent_dim=8):
        super(Autoencoder, self).__init__()
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.append(nn.Dense(prev_dim, h_dim, weight_init=HeNormal()))
            encoder_layers.append(nn.ReLU())
            prev_dim = h_dim
        encoder_layers.append(nn.Dense(prev_dim, latent_dim, weight_init=HeNormal()))
        self.encoder = nn.SequentialCell(encoder_layers)

        decoder_layers = []
        prev_dim = latent_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.append(nn.Dense(prev_dim, h_dim, weight_init=HeNormal()))
            decoder_layers.append(nn.ReLU())
            prev_dim = h_dim
        decoder_layers.append(nn.Dense(prev_dim, input_dim, weight_init=HeNormal()))
        self.decoder = nn.SequentialCell(decoder_layers)

    def construct(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

def train_autoencoder(train_images, epochs=50, lr=0.001, batch_size=128):
    """训练自编码器，返回 (encoder, scaler_for_ae)（与实验三基线一致）"""
    scaler_for_ae = MinMaxScaler(feature_range=(0, 1))
    train_images_scaled = scaler_for_ae.fit_transform(train_images)

    train_dataset = ds.NumpySlicesDataset(train_images_scaled, column_names=["image"], shuffle=True).batch(batch_size)

    autoencoder = Autoencoder(input_dim=train_images.shape[1],
                              hidden_dims=AUTOENCODER_HIDDEN_DIMS,
                              latent_dim=PCA_COMPONENTS)
    loss_fn = nn.MSELoss()
    optimizer = nn.Adam(autoencoder.trainable_params(), learning_rate=lr)

    def forward_fn(data):
        encoded, decoded = autoencoder(data)
        loss = loss_fn(decoded, data)
        return loss

    grad_fn = ms.value_and_grad(forward_fn, None, optimizer.parameters)

    print("开始训练自编码器...")
    for epoch in range(epochs):
        total_loss = 0
        step = 0
        for data in train_dataset.create_tuple_iterator():
            data = data[0]
            loss, grads = grad_fn(data)
            optimizer(grads)
            total_loss += loss.asnumpy()
            step += 1
        avg_loss = total_loss / step
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")

    return autoencoder.encoder, scaler_for_ae

def preprocess_data_autoencoder(images, encoder=None, scaler_for_ae=None, final_scaler=None, is_train=True):
    """
    自编码器降维 + MinMax(-1,1) 预处理函数（与实验三基线一致）
    返回 numpy 特征 (N, 8)；训练集训练自编码器，测试集复用
    """
    if is_train:
        encoder, scaler_for_ae = train_autoencoder(images, epochs=AUTOENCODER_EPOCHS,
                                                    lr=AUTOENCODER_LR, batch_size=128)
        images_scaled = scaler_for_ae.transform(images)
        images_tensor = Tensor(images_scaled, ms.float32)
        latent_features = encoder(images_tensor).asnumpy()
        print(f"自编码器降维完成，输出形状: {latent_features.shape}")

        final_scaler = MinMaxScaler(feature_range=(-1, 1))
        features = final_scaler.fit_transform(latent_features)
    else:
        images_scaled = scaler_for_ae.transform(images)
        images_tensor = Tensor(images_scaled, ms.float32)
        latent_features = encoder(images_tensor).asnumpy()
        features = final_scaler.transform(latent_features)

    return features, final_scaler, encoder, scaler_for_ae


In [9]:
# 第四部分：SVM 训练与评估函数
# ============================================

def train_evaluate_svm(train_features, train_labels, test_features, test_labels,
                       kernel='linear', C=1.0):
    """训练SVM，返回 (模型, 训练集准确率, 测试集准确率)"""
    print(f"\n训练 SVM (kernel={kernel}, C={C}) ...")
    model = SVC(kernel=kernel, C=C, random_state=SEED)
    model.fit(train_features, train_labels)

    train_pred = model.predict(train_features)
    test_pred = model.predict(test_features)

    train_acc = (train_pred == train_labels).mean()
    test_acc = (test_pred == test_labels).mean()

    print(f"  训练集准确率: {train_acc:.4f}")
    print(f"  测试集准确率: {test_acc:.4f}")

    return model, train_acc, test_acc


In [10]:
# 第五部分：主程序 - 自编码器降维 + 线性SVM 经典对照组
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print("自编码器降维 + 线性SVM 经典对照组")
    print("=" * 60)

    # 1. 加载数据
    print("\n步骤1：加载MNIST数据")
    all_images, all_labels = load_mnist_data()
    print(f"数据大小: {len(all_images)}")

    # 2. 划分数据集（与量子实验一致）
    print("\n步骤2：划分训练集和测试集")
    train_images, test_images, train_labels, test_labels = train_test_split(
        all_images, all_labels, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
    print(f"训练集: {len(train_images)} 样本, 测试集: {len(test_images)} 样本")

    # 3. 自编码器降维 + 归一化
    print("\n步骤3：自编码器降维预处理")
    train_features, final_scaler, encoder, scaler_for_ae = preprocess_data_autoencoder(
        train_images, is_train=True
    )
    test_features, _, _, _ = preprocess_data_autoencoder(
        test_images, encoder=encoder, scaler_for_ae=scaler_for_ae, final_scaler=final_scaler, is_train=False
    )
    print(f"特征形状 - 训练集: {train_features.shape}, 测试集: {test_features.shape}")

    # 4. 训练并评估 SVM
    print("\n步骤4：训练并评估线性SVM")
    model, train_acc, test_acc = train_evaluate_svm(
        train_features, train_labels, test_features, test_labels,
        kernel=SVM_KERNEL, C=SVM_C
    )

    # 5. 结果汇总
    print("\n" + "=" * 60)
    print("自编码器 + 线性SVM 对照组结果汇总")
    print("=" * 60)
    print(f"核函数: {SVM_KERNEL}, C: {SVM_C}")
    print(f"训练集准确率: {train_acc:.4f}")
    print(f"测试集准确率: {test_acc:.4f}")

    # 6. 保存结果
    np.savez('autoencoder_svm_control_results.npz',
             kernel=SVM_KERNEL,
             C=SVM_C,
             train_acc=train_acc,
             test_acc=test_acc)
    print("\n结果已保存到 autoencoder_svm_control_results.npz")


自编码器降维 + 线性SVM 经典对照组

步骤1：加载MNIST数据
正在加载MNIST数据集...
总样本数: 70000, 采样后: 70000
类别分布: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64), array([6903, 7877, 6990, 7141, 6824, 6313, 6876, 7293, 6825, 6958],
      dtype=int64))
数据大小: 70000

步骤2：划分训练集和测试集
训练集: 56000 样本, 测试集: 14000 样本

步骤3：自编码器降维预处理
开始训练自编码器...
Epoch 1/50, Loss: 0.051426
Epoch 10/50, Loss: 0.023103
Epoch 20/50, Loss: 0.020723
Epoch 30/50, Loss: 0.019618
Epoch 40/50, Loss: 0.018969
Epoch 50/50, Loss: 0.018645
自编码器降维完成，输出形状: (56000, 8)
特征形状 - 训练集: (56000, 8), 测试集: (14000, 8)

步骤4：训练并评估线性SVM

训练 SVM (kernel=linear, C=1.0) ...
  训练集准确率: 0.9194
  测试集准确率: 0.9112

自编码器 + 线性SVM 对照组结果汇总
核函数: linear, C: 1.0
训练集准确率: 0.9194
测试集准确率: 0.9112

结果已保存到 autoencoder_svm_control_results.npz
